# 🧬 Structured Output: Schemas Instead of Strings

## Learning Objectives
In this notebook, you will learn:
1. **`with_structured_output`** - make a model return a typed object instead of prose
2. **Three schema styles** - Pydantic, `TypedDict`, and dataclasses, and when each fits
3. **`include_raw=True`** - keep the original `AIMessage` alongside the parsed result
4. **Nested schemas** - lists, optional fields, and models inside models
5. **`response_format` on agents** - the same idea at the agent level, via `structured_response`

## Prerequisites
- Completed `1-langchainintro.ipynb` through `4-messages.ipynb`
- `pip install langchain langchain-openai pydantic python-dotenv`
- A `.env` file with `OPENAI_API_KEY`

---
## 💡 Part 1: Why Structured Output?

A model can be asked to return a response matching a given **schema** rather than free text.
That guarantees the output parses cleanly into your downstream code — no regexes over prose,
no "sometimes it adds a preamble" bugs.

### Key Concepts:
- **The schema goes to the provider**, not into the prompt as an instruction. Modern providers
  constrain decoding to the schema, so malformed output is not merely unlikely — it is
  structurally prevented.
- **`.with_structured_output(Schema)`** returns a *new* runnable whose `.invoke()` gives you an
  instance of `Schema` instead of an `AIMessage`.
- **Three schema styles** are supported: Pydantic (validation + descriptions), `TypedDict`
  (plain typing, no runtime validation), and dataclasses.

---
## 🔑 Part 2: Environment Setup

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load credentials and initialize the model
# ============================================================================
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "❌ Set OPENAI_API_KEY in your .env file"

model = init_chat_model("gpt-4.1")

print(f"✅ Environment loaded — using {type(model).__name__}")

---
## 🐍 Part 3: Pydantic Schemas

Pydantic models give the richest feature set: field validation, descriptions, defaults, and
nested structures.

### Key Insight:
`Field(description=...)` is **not** documentation for you — it is shipped to the model as part
of the schema. A field described as `"The movie's rating out of 10"` gets a 0–10 number; the
same field with no description gets whatever scale the model guesses. Descriptions are prompt
engineering.

In [ ]:
# ============================================================================
# SCHEMA DEFINITION: A flat Pydantic model
# ============================================================================
from pydantic import BaseModel, Field


class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")


print("✅ Schema defined:", list(Movie.model_fields))

In [ ]:
# ============================================================================
# BINDING: with_structured_output returns a NEW runnable
# ============================================================================
model_with_structure = model.with_structured_output(Movie)
model_with_structure

### 3.1 🔍 Unstructured vs. Structured — the Same Question

Run both and compare the return types. The plain call gives an `AIMessage` whose `.content` is
a paragraph you would have to parse; the structured call gives a `Movie` instance with typed
attributes.

In [ ]:
# ============================================================================
# BASELINE: Plain invocation returns prose in an AIMessage
# ============================================================================
model.invoke("Provide details about the movie Inception")

In [ ]:
# ============================================================================
# STRUCTURED: Same prompt, a typed Movie object comes back
# ============================================================================
response = model_with_structure.invoke("Provide details about the movie Inception")

print(f"📋 type:     {type(response).__name__}")
print(f"🎬 title:    {response.title}")
print(f"📅 year:     {response.year}")
print(f"🎥 director: {response.director}")
print(f"⭐ rating:   {response.rating}")

response

### 3.2 📦 Keeping the Raw Message Too

`include_raw=True` changes the return value to a dict with three keys:

- **`raw`** — the original `AIMessage`, including `usage_metadata` and provider metadata
- **`parsed`** — your schema instance
- **`parsing_error`** — populated instead of raising, if the output could not be coerced

Use it in production: you keep token counts for cost tracking, and a parse failure becomes a
value you can branch on rather than an exception that kills the request.

In [ ]:
# ============================================================================
# INCLUDE_RAW: Get the AIMessage and the parsed object together
# ============================================================================
class Movie(BaseModel):
    """A movie with details."""

    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")


model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")

print(f"📋 keys:          {list(response)}")
print(f"🎬 parsed:        {response['parsed']}")
print(f"⚠️  parsing_error: {response['parsing_error']}")
print(f"📊 tokens:        {response['raw'].usage_metadata}")

response

### 3.3 🪆 Nested Structures

Schemas compose. A field can be a list of other models, an optional value, or both — and the
provider constrains the whole tree, not just the top level.

Note `budget: float | None = Field(None, ...)`: an optional field the model may legitimately
omit, which is the right way to model "this data might not exist" instead of forcing the model
to invent a number.

In [ ]:
# ============================================================================
# NESTED SCHEMA: Models inside models, lists, and optional fields
# ============================================================================
class Actor(BaseModel):
    name: str
    role: str


class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]                # a list of nested models
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")

print(f"🎬 {response.title} ({response.year})")
print(f"🎭 genres: {', '.join(response.genres)}")
for actor in response.cast:
    print(f"   👤 {actor.name} as {actor.role}")

response

---
## 📘 Part 4: `TypedDict` Schemas

`TypedDict` is a simpler alternative built on Python's own typing module — ideal when you do
**not** need runtime validation. The result comes back as a plain `dict`, not an object.

To attach a description to a field, use `Annotated[type, default, "description"]`. The middle
`...` means "required".

> **Note**: `TypedDict` performs no validation. If a provider returns a string where you
> declared an `int`, Pydantic would raise; `TypedDict` hands you the string. Choose Pydantic
> when the data crosses a trust boundary.

In [ ]:
# ============================================================================
# TYPEDDICT SCHEMA: Annotated[type, ..., "description"] carries field docs
# ============================================================================
from typing_extensions import Annotated, NotRequired, TypedDict


class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_with_typeddict = model.with_structured_output(MovieDict)

response = model_with_typeddict.invoke("Please provide the details of the movie Avengers")

print(f"📋 type: {type(response).__name__}  <- a plain dict, not an object")
response

#### ⚠️ Optional Fields in a `TypedDict`

A `TypedDict` **cannot** take a default value the way a Pydantic model can. Writing
`budget: float | None = Field(None, ...)` inside a `TypedDict` is silently ignored — Python
keeps the annotation and throws the assignment away, so the description never reaches the
model and the field stays required.

The correct spelling is `NotRequired[...]`, optionally wrapped in `Annotated` to keep the
description:

In [ ]:
# ============================================================================
# NESTED TYPEDDICT: NotRequired for optional fields (NOT `= Field(...)`)
# ============================================================================
class ActorDict(TypedDict):
    name: Annotated[str, ..., "The actor's name"]
    role: Annotated[str, ..., "The character they play"]


class MovieDetailsDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    cast: Annotated[list[ActorDict], ..., "The main cast"]
    genres: Annotated[list[str], ..., "Genres the movie belongs to"]
    # Optional field: NotRequired, not a default value.
    budget: NotRequired[Annotated[float, "Budget in millions USD"]]


model_with_structure = model.with_structured_output(MovieDetailsDict)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

### 4.1 🔎 Checking Provider Support

Not every model supports structured output, and the mechanism differs by provider. `.profile`
(new in LangChain 1.1) reports what the model actually supports — sourced from the open
`models.dev` project — so you can branch on capability instead of guessing.

In [ ]:
# ============================================================================
# MODEL PROFILE: Capability introspection, printed as readable JSON
# ============================================================================
import json

profile = model.profile

# `profile` is a plain dict (a ModelProfile TypedDict). `default=str` guards
# against any non-JSON-native values a provider might include.
print(json.dumps(profile, indent=2, default=str, sort_keys=True))

---
## 🏗️ Part 5: Dataclass Schemas and Agent `response_format`

A dataclass is a class that mainly holds data, created with the `@dataclass` decorator. It is
the third supported schema style — no extra dependency, no validation.

More importantly, this section shows structured output at the **agent** level. Instead of
`.with_structured_output()` on a model, you pass `response_format=Schema` to `create_agent`,
and the parsed object lands in `result["structured_response"]`.

### Key Concepts:
- **`response_format`** auto-selects a strategy (`ProviderStrategy` where the provider supports
  native structured output)
- **`result["structured_response"]`** holds the typed result; `result["messages"]` still holds
  the full conversation
- **All three styles work** — Pydantic, `TypedDict`, and dataclass — so pick per use case

In [ ]:
# ============================================================================
# AGENT + PYDANTIC: response_format on create_agent
# ============================================================================
from langchain.agents import create_agent

EXTRACTION_PROMPT = "Extract contact info from: John Doe, john@example.com, (555) 123-4567"


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo,  # auto-selects ProviderStrategy
)

result = agent.invoke({"messages": [{"role": "user", "content": EXTRACTION_PROMPT}]})
result

In [ ]:
# ============================================================================
# READING THE RESULT: the parsed object lives under "structured_response"
# ============================================================================
result["structured_response"]

In [ ]:
# ============================================================================
# AGENT + TYPEDDICT: same agent, a dict-shaped schema
# ============================================================================
class ContactInfoDict(TypedDict):
    """Contact information for a person."""

    name: Annotated[str, ..., "The name of the person"]
    email: Annotated[str, ..., "The email address of the person"]
    phone: Annotated[str, ..., "The phone number of the person"]


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfoDict,
)

result = agent.invoke({"messages": [{"role": "user", "content": EXTRACTION_PROMPT}]})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [ ]:
# ============================================================================
# AGENT + DATACLASS: the third schema style
# ============================================================================
from dataclasses import dataclass


@dataclass
class ContactInfoDataclass:
    """Contact information for a person."""

    name: str    # The name of the person
    email: str   # The email address of the person
    phone: str   # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfoDataclass,
)

result = agent.invoke({"messages": [{"role": "user", "content": EXTRACTION_PROMPT}]})

result["structured_response"]

---
## 📝 Summary

In this notebook, we learned:

### 1. Structured Output Basics
- **`.with_structured_output(Schema)`** returns a new runnable that yields typed objects
- **The schema reaches the provider**, which constrains decoding — malformed output is
  prevented, not just discouraged
- **`Field(description=...)` is prompt engineering**: it tells the model what the field means

### 2. Choosing a Schema Style
- **Pydantic**: validation, descriptions, defaults, nesting — the default choice, and the only
  one that catches a provider returning the wrong type
- **`TypedDict`**: plain typing, returns a `dict`, no runtime validation — use `NotRequired[...]`
  for optional fields, never `= Field(...)`, which is silently discarded
- **Dataclass**: no extra dependency, no validation

### 3. Production Details
- **`include_raw=True`** returns `{"raw", "parsed", "parsing_error"}` — keeps token counts and
  turns parse failures into a value instead of an exception
- **`.profile`** reports whether the model supports structured output at all

### 4. At the Agent Level
- **`response_format=Schema`** on `create_agent` does the same job for a whole agent
- **`result["structured_response"]`** holds the parsed object; `result["messages"]` still has
  the full trail

### Next Steps
- **`6-middleware.ipynb`** — intercept the agent loop with summarization, PII redaction, call
  limits, and human-in-the-loop approval